# Economies of Space Hackathon 2026
This notebook contains some useful functions that should ease using the large HGB corpus.
It is split into different topics, not intended to be run in one sequence.

This notebook only concerns the XML dataset, not the graph database.

## Accessing the Data
To access the XML corpus, we recommend either downloading [the zipped XML file](https://drive.switch.ch/index.php/s/HPlpyYeXUdx2Q92), or grabbing the XML files from our postgresql-db.

### PostgreSQL
The advantage of this approach is that it allows you to pre-select documents on certain metadata information. See [here](https://github.com/history-unibas/economies-of-space-database) for all info about the db and attributes to filter by. Remember that only the years 1400-1700 and only german-language and mixed-language documents were processed, but the table contains more documents than that.

Mind you the corpus is very large. The code below implements a method to stream documents one by one instead of loading in the whole corpus at once. If possible, use this to create a subcorpus with only the information you're interested in that can then be processed much faster in the future.

In [ ]:
# install modules
%pip install -q psycopg2
%pip install -q pandas
%pip install -q lxml

In [104]:
from lxml import etree

def filter_by_xml(doc):
    """
    Use this function to filter the XMLs received from the DB based on XML-contents (e.g. span or event information)
    """
    # Example filter: Only documents with at least one property purchase!
    matches = doc.xpath("./eventGroups/eventGroup[@class='property-purchase']")
    if len(matches) > 0:
        return True

    return False

In [ ]:
import psycopg2
import pandas as pd

DB_NAME = "hgb"
DB_USER = "read_only"
DB_PASSWORD = "read_only"
DB_HOST = "130.92.252.51"
DB_PORT = "8241"
XML_COLUMN_NAME = "annotationautomated" 

conn = psycopg2.connect(
    dbname=DB_NAME,
    user=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
    sslmode='require'
)

# Use WHERE-Filtering to filter by {year, language, source}
# or join with dossier and series tables to filter by house or street
query = f"""
    SELECT entryid, {XML_COLUMN_NAME} 
    FROM project_entry
    WHERE (language = 'german' OR language = 'mixed') AND
    year >= 1400 AND year <= 1700;
"""

# this loads only CHUNK_SIZE rows at once into memory
CHUNK_SIZE = 1000
df_iterator = pd.read_sql(query, conn, chunksize=CHUNK_SIZE)

filtered_results = []

print("Starting streaming analysis...")

# Iterate through the database, chunk by chunk
for chunk_idx, df_chunk in enumerate(df_iterator):
    print(f"  Processing chunk {chunk_idx + 1}...")
    
    # Iterate through the individual rows in the current chunk
    for index, row in df_chunk.iterrows():
        xml_string = row[XML_COLUMN_NAME]
        
        # catches everything non-processed
        if not xml_string:
            continue
            
        try:
            root = etree.fromstring(xml_string.encode('utf-8'))
            filter = filter_by_xml(root)
            
            if filter:
                filtered_results.append(xml_string)
                
        except etree.XMLSyntaxError:
            print(f"Warning: Could not parse XML for entry {row['entryid']}")

# Close the database connection
conn.close()

print(f"\nProcessing complete! Found {len(filtered_results)} documents matching the XML filter.")

# Display the top results
filtered_results[:5]

### Working with the ZIP-Download
Those who opt for the ZIP download can parse the XML directly with lxml (after extracting it). Due to its size, iterparse() should be used. With the following code, only one document is loaded into memory at a time. Here too, it is recommended to create a filter suited to your own research question and to save the searched documents, so that you can then continue working with a smaller corpus.

In [ ]:
from lxml import etree

def filter_by_xml(doc):
    """
    Use this function to filter the XMLs received from the DB based on XML-contents (e.g. span or event information)
    """
    # Example filter: Only documents with at least one property purchase!
    matches = doc.xpath("./eventGroups/eventGroup[@class='property-purchase']")
    if len(matches) > 0:
        return True

    return False

In [ ]:
from lxml import etree

INPUT_XML = "hgb_full_26_05_29_05.xml"
filtered_results = []

print(f"Streaming XML from {INPUT_XML}...")
    
# Setup the streaming parser targeting <document> elements
context = etree.iterparse(INPUT_XML, events=('end',), tag='document')
        
for _, doc in context:
    
    # apply filter function defined in the previous cell
    filter = filter_by_xml(doc)
    if filter:
        filtered_results.append(etree.tostring(doc, encoding="utf8").decode("utf8"))
        
    # Clear the element from memory
    doc.clear()
    while doc.getprevious() is not None:
        del doc.getparent()[0]

print(f"\nProcessing complete! Found {len(filtered_results)} documents matching the XML filter.")

# Display the top results
filtered_results[:5]

### Saving the Filtered Documents
We write a new corpus file here, only including the documents that are of interest. (of course you can also keep them in memory for now and continue working like that).

In [ ]:
OUTPUT_XML = "filtered_corpus.xml"

with etree.xmlfile(OUTPUT_XML, encoding='utf-8') as xf:
    xf.write_declaration(version="1.0")

    with xf.element("corpus", {}):
        for doc in filtered_results:
            xf.write(etree.fromstring(doc))

## Parsing the XML: Core Loop
The following code cell uses iterparse() to iterate the XML (either a filtered one or the full corpus) in an efficient manner.

It includes a usage example where we get some info to analyze (occupations of actors in the previously filtered purchasing events) and save it to a pandas dataframe including the metadata.

In [ ]:
from lxml import etree

INPUT_XML = "filtered_corpus.xml"
records = []

print(f"Streaming XML from {INPUT_XML}...")
    
# Setup the streaming parser targeting <document> elements
context = etree.iterparse(INPUT_XML, events=('end',), tag='document')
        
for _, doc in context:
    

    ### EXAMPLE USAGE

    # we can apply another filter here if we need to
    filter = True  # Placeholder
    if not filter:
        continue

    # get the metadata
    metadata_elem = doc.find("./metadata")
    record = dict(metadata_elem.attrib)

    # add information to analyse
    spans_elem = doc.find("./spans")
    purchasing_events = doc.xpath("./eventGroups/eventGroup[@class='property-purchase']/event")
    buyer_occupations = []
    seller_occupations = []
    for purchase_event in purchasing_events:
        actors = purchase_event.xpath("./role[@role='buyer' or @role='seller']")
        # because role elements only contain a ref-attribute pointing to the span in the text, we need to do this:
        for actor in actors:
            actor_id = actor.get("ref")
            actor_class = actor.get("role")
            actor_span = spans_elem.find(f".//span[@id='{actor_id}']")
            # check that actor is a person - organizations don't have occupations
            if actor_span.get("class") != "per":
                continue
            # occupation information is given as part of the head - and occupation descriptions are usually appositions
            occupation_spans = actor_span.xpath("./span[@element='appo']/span[@element='head' and @class='occ']")  # we account for multiple occ infos for one person
            for occupation_span in occupation_spans:
                normalization = occupation_span.get("norm", "unk")
                if actor_class == "buyer":
                    buyer_occupations.append(normalization)
                else:
                    seller_occupations.append(normalization)
    record["buyers"] = buyer_occupations
    record["sellers"] = seller_occupations

    records.append(record)
        
    ### EXAMPLE END
        
    # Clear the element from memory
    doc.clear()
    while doc.getprevious() is not None:
        del doc.getparent()[0]


# convert our collected data into a dataframe for easy further work
records_df = pd.DataFrame.from_records(records)

records_df

This data can now easily be input into typical plotting libraries or exported to be used in other software.

In [ ]:
%pip install -q matplotlib

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 1. Convert 'year' to integer to prevent index mismatches
records_df['year'] = records_df['year'].astype(int)

# 2. Explode the lists and filter out "unk"
buyers_exp = records_df[['year', 'buyers']].explode('buyers').dropna()
sellers_exp = records_df[['year', 'sellers']].explode('sellers').dropna()

buyers_exp = buyers_exp[buyers_exp['buyers'] != 'unk']
sellers_exp = sellers_exp[sellers_exp['sellers'] != 'unk']

# 3. Identify the top 5 most common occupations
top_buyers = buyers_exp['buyers'].value_counts().head(5).index
top_sellers = sellers_exp['sellers'].value_counts().head(5).index

# 4. Filter and group by year
buyers_trend = (buyers_exp[buyers_exp['buyers'].isin(top_buyers)]
                .groupby(['year', 'buyers']).size().unstack(fill_value=0))

sellers_trend = (sellers_exp[sellers_exp['sellers'].isin(top_sellers)]
                 .groupby(['year', 'sellers']).size().unstack(fill_value=0))

# 5. FIX THE X-AXIS: Create a common timeline of all years
min_year = records_df['year'].min()
max_year = records_df['year'].max()
all_years = range(min_year, max_year + 1)

# Reindex to fill any entirely missing years with 0
buyers_trend = buyers_trend.reindex(all_years, fill_value=0)
sellers_trend = sellers_trend.reindex(all_years, fill_value=0)

# smoothing 5-years
buyers_smoothed = buyers_trend.rolling(window=5, min_periods=1).mean()
sellers_smoothed = sellers_trend.rolling(window=5, min_periods=1).mean()

# 6. Plot vertically
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 12))

buyers_smoothed.plot(kind='area', stacked=True, ax=ax1, title='Top Buyer Occupations Over Time')
ax1.set_ylabel('Count')
ax1.set_xlim(min_year, max_year) 

sellers_smoothed.plot(kind='area', stacked=True, ax=ax2, title='Top Seller Occupations Over Time')
ax2.set_ylabel('Count')
ax2.set_xlim(min_year, max_year) 

plt.tight_layout()
plt.show()

## Other Useful Functions and Resources
Here are some potentially useful resources and the functions to use them in your notebook that we created for previous uses.

### Currency Conversion
This cell loads our currency conversion table and provides a function to quickly convert any monetary values from the XML.

It uses some interpolation to fill unknown values, but you may need to expand the table (and/or the interpolation) depending on your analysis.

In [ ]:
import pandas as pd

currency_conversion_df = pd.read_csv("resources/currency_conversion.tsv", delimiter="\t")

currency_conversion_df['currency'] = (
    currency_conversion_df['Mass'].fillna('').astype(str) + " " + 
    currency_conversion_df['Währung / Gut'].fillna('').astype(str)
).str.strip()
currency_conversion_df['Umrechnung zu Pfund'] = currency_conversion_df['Umrechnung zu Pfund'].astype(str)
currency_conversion_df['Umrechnung zu Pfund'] = currency_conversion_df['Umrechnung zu Pfund'].str.replace(',', '.', regex=False)
currency_conversion_df['Umrechnung zu Pfund'] = pd.to_numeric(currency_conversion_df['Umrechnung zu Pfund'], errors='coerce')

currency_conversion_df['year_numeric'] = currency_conversion_df['Jahr'].str.split('-').str[0].astype(int)

interpolated_df = currency_conversion_df.sort_values(by=['currency', 'year_numeric'])

def interpolate_robustly(group):
    valid_points = group.count()
    if valid_points < 3:
        result = group.interpolate(method='linear', limit_direction='both')
        if valid_points == 1:
             return result.ffill().bfill()
        
        return result

    try:
        return group.interpolate(method='spline', order=1, limit_direction='both')
    except:
        # Safety net: If spline fails (e.g. duplicates or vertical lines), use linear
        return group.interpolate(method='linear', limit_direction='both')

interpolated_df['Umrechnung zu Pfund'] = interpolated_df.groupby('currency')['Umrechnung zu Pfund'].transform(interpolate_robustly)
interpolated_df['decade_key'] = interpolated_df['year_numeric'].astype(str).str[:3]

conversion_lookup = interpolated_df.set_index(['currency', 'decade_key'])['Umrechnung zu Pfund'].to_dict()

def convert_amount(data_tuple):
    year, amount, currency = data_tuple
    currency = currency.replace("-", " ")

    # catch to deal with 'Stebler'
    currency = currency.replace("Stebler", "").strip()

    if currency == 'Pfund':
        return 1.0
    
    lookup_key = str(year)[:3]
    
    rate = conversion_lookup.get((currency, lookup_key))
    
    if rate is None:
        if currency.split(" ")[0].strip() in ["Schilling", "Pfund"]:
            currency = currency.split(" ")[0].strip()
            rate = conversion_lookup.get((currency, lookup_key))
            if rate is None:
                print(f"Warning: No rate found for {currency} in decade {lookup_key}X")
                return 0
        else:
            print(f"Warning: No rate found for {currency} in decade {lookup_key}X")
            return 0

    if not pd.notna(rate):
        print(f"Warning: Rate is NaN for {currency} in decade {lookup_key}X")
        return 0  # NOTE: You need to decide how to handle missing conversion values
    else:
        return amount * rate

# --- Usage Example ---
rate_1508 = convert_amount((1508, 100.0, "Becher-Bohnen")) 
print(f"Rate for 1508: {rate_1508}")

rate_1523 = convert_amount((1523, 100.0, "Gulden"))
print(f"Rate for 1523: {rate_1523}")

rate_1523 = convert_amount((1523, 100.0, "Schilling-Pfennig"))
print(f"Rate for 1523: {rate_1523}")

rate_1638 = convert_amount((1638, 100.0, "Schilling-Stebler"))
print(f"Rate for 1638: {rate_1638}")

### Get Guild, Trade Information for Occupations
This cell loads a dictionary that maps occupation normalizations to their guilds and their trade (partially based on Fueglister 1981).

Note that not all occupations have been mapped to guilds.

Freie Berufe, Verwaltung, and some minor groups were introduced in addition to guilds.

In [ ]:
import pandas as pd

zunft_df = pd.read_csv("resources/zuenfte.tsv", sep="\t")
zunft_exploded_df = zunft_df.assign(
    Norm=zunft_df['Normvarianten'].str.lower().str.split(';')
).explode('Norm')

zunft_lookup = zunft_exploded_df.set_index('Norm')['Zunft'].to_dict()
gewerbe_lookup = zunft_exploded_df.set_index('Norm')['Gewerbe'].to_dict()

zunft_lookup

In [ ]:
# USAGE_EXAMPLE: Doing the plot from above but for guilds.
import pandas as pd
import matplotlib.pyplot as plt

# 1. Convert 'year' to integer to prevent index mismatches
records_df['year'] = records_df['year'].astype(int)

# 2. Explode the lists and filter out "unk"
buyers_exp = records_df[['year', 'buyers']].explode('buyers').dropna()
sellers_exp = records_df[['year', 'sellers']].explode('sellers').dropna()

buyers_exp = buyers_exp[buyers_exp['buyers'] != 'unk']
sellers_exp = sellers_exp[sellers_exp['sellers'] != 'unk']

### CHANGE: Applies the guild mapping
buyers_exp['buyers'] = buyers_exp['buyers'].map(zunft_lookup).fillna(buyers_exp['buyers'])
sellers_exp['sellers'] = sellers_exp['sellers'].map(zunft_lookup).fillna(sellers_exp['sellers'])

# 3. Identify the top 5 most common occupations
top_buyers = buyers_exp['buyers'].value_counts().head(5).index
top_sellers = sellers_exp['sellers'].value_counts().head(5).index

# 4. Filter and group by year
buyers_trend = (buyers_exp[buyers_exp['buyers'].isin(top_buyers)]
                .groupby(['year', 'buyers']).size().unstack(fill_value=0))

sellers_trend = (sellers_exp[sellers_exp['sellers'].isin(top_sellers)]
                 .groupby(['year', 'sellers']).size().unstack(fill_value=0))

# 5. FIX THE X-AXIS: Create a common timeline of all years
min_year = records_df['year'].min()
max_year = records_df['year'].max()
all_years = range(min_year, max_year + 1)

# Reindex to fill any entirely missing years with 0
buyers_trend = buyers_trend.reindex(all_years, fill_value=0)
sellers_trend = sellers_trend.reindex(all_years, fill_value=0)

# smoothing 5-years
buyers_smoothed = buyers_trend.rolling(window=5, min_periods=1).mean()
sellers_smoothed = sellers_trend.rolling(window=5, min_periods=1).mean()

# 6. Plot vertically
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 12))

buyers_smoothed.plot(kind='area', stacked=True, ax=ax1, title='Top Buyer Occupations Over Time')
ax1.set_ylabel('Count')
ax1.set_xlim(min_year, max_year) 

sellers_smoothed.plot(kind='area', stacked=True, ax=ax2, title='Top Seller Occupations Over Time')
ax2.set_ylabel('Count')
ax2.set_xlim(min_year, max_year) 

plt.tight_layout()
plt.show()

# Creating Maps
Coordinates for each property are available as part of the metadata. The following cell shows how to read these coordinates and use them as part of a script to create a simple map. There are multiple approaches on how to do this, but using geopandas is pretty straightforward.

In [ ]:
%pip install -q geopandas
%pip install -q folium

In [ ]:
import geopandas as gpd

map_df = gpd.GeoDataFrame(records_df)
map_df['geometry'] = gpd.GeoSeries.from_wkt(map_df['location'], on_invalid='ignore')
map_df = map_df.dropna(subset=['geometry'])  # ignore invalid coordinate data

In [ ]:
import folium
import pandas as pd
from pyproj import Transformer
import branca.colormap as cm

map_df['coord_x'] = map_df.geometry.x
map_df['coord_y'] = map_df.geometry.y

# Transform the coordinate format
transformer = Transformer.from_crs(2056, 4326)
map_df['lat'], map_df['lon'] = zip(*map_df.apply(
    lambda row: transformer.transform(row['coord_x'], row['coord_y']), axis=1
))

# EXAMPLE: Make points redder if more purchases happened
distinct_df = map_df.groupby(['dossierid']).agg(
    id=('dossierid', 'first'),
    lat=('lat', 'first'),
    lon=('lon', 'first'),
    mention_count=('dossierid', 'size') 
).reset_index()

# Prepare the color scaling
min_count = 1
max_count = 20

colormap = cm.LinearColormap(colors=['blue', 'yellow', 'red'], vmin=min_count, vmax=max_count)
colormap.caption = "Number of Mentions"

# Initialize the Map
m = folium.Map(
    location=[distinct_df['lat'].mean(), distinct_df['lon'].mean()], 
    zoom_start=14, 
    control_scale=True
)

# Add the historical map
folium.WmsTileLayer(
    url='https://wms.geo.bs.ch/',
    layers='HP_Situationsplan_Basel_1862',
    attr='Geodaten Kanton Basel-Stadt',
    name='Basel 1862'
).add_to(m)

m.add_child(colormap)

# Plot the distinct Points
for index, row in distinct_df.iterrows():
    count_val = row['mention_count']
    
    popup_html = f"""
    <div style="width: 150px">
        <b> {row['id']} </b><lb/>
        <b>Mentions:</b> {int(count_val)}
    </div>
    """
    
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=5,
        color=colormap(count_val),
        fill=True,
        fill_color=colormap(count_val),
        fill_opacity=0.8,
        weight=1,
        popup=folium.Popup(popup_html, max_width=200)
    ).add_to(m)

# Save
output_file = "frequency_map.html"
m.save(output_file)

print(f"Map saved to {output_file}. Processed {len(distinct_df)} distinct locations.")